# Thí nghiệm cấu hình A — So sánh 4 mô hình hồi quy

## Params

In [ ]:
MODEL_PARAMS = {
    "Linear Regression": {
        "fit_intercept": True,
    },
    "Random Forest": {
        "n_estimators": 200,
        "max_depth": 12,
        "min_samples_leaf": 5,
        "random_state": 42,
        "n_jobs": -1,
    },
    "Gradient Boosting": {
        "n_estimators": 200,
        "learning_rate": 0.05,
        "max_depth": 3,
        "min_samples_leaf": 5,
        "loss": "squared_error",
        "random_state": 42,
    },
    "MLP": {
        "hidden_layer_sizes": (64, 32),
        "activation": "relu",
        "solver": "adam",
        "alpha": 0.0001,
        "learning_rate_init": 0.001,
        "batch_size": 128,
        "max_iter": 500,
        "random_state": 42,
    },
}

CALENDAR_COLUMNS = ["hour", "day_of_week", "month"]
BINARY_COLUMNS = ["is_weekend", "is_holiday"]
TARGET_COLUMN = "traffic_volume"
CLIP_NEGATIVE_PREDICTIONS = True
HOURS_TO_PLOT = 72

## Đọc dữ liệu và chọn đặc trưng cấu hình A

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

cwd = Path.cwd()
project_dir = next(
    (
        candidate
        for parent in (cwd, *cwd.parents)
        for candidate in (parent, parent / "metro_interstate_traffic_volume")
        if (candidate / "data" / "processed" / "train.csv").is_file()
    ),
    None,
)
if project_dir is None:
    raise FileNotFoundError("Không tìm thấy data/processed/train.csv.")

processed_dir = project_dir / "data" / "processed"
train = pd.read_csv(processed_dir / "train.csv", parse_dates=["date_time"])
validation = pd.read_csv(processed_dir / "validation.csv", parse_dates=["date_time"])

assert train.columns.equals(validation.columns)
for part in (train, validation):
    assert part["date_time"].is_unique
    assert part["date_time"].is_monotonic_increasing
assert train["date_time"].max() < validation["date_time"].min()

holiday_columns = [column for column in train.columns if column.startswith("holiday_")]
binary_columns = BINARY_COLUMNS + holiday_columns
features_A = CALENDAR_COLUMNS + binary_columns

for part in (train, validation):
    assert part[features_A + [TARGET_COLUMN]].notna().all().all()

X_train, y_train = train[features_A], train[TARGET_COLUMN]
X_validation, y_validation = validation[features_A], validation[TARGET_COLUMN]

## Huấn luyện bốn mô hình trên tập train

In [ ]:
preprocessor = ColumnTransformer([
    (
        "calendar",
        OneHotEncoder(handle_unknown="ignore", sparse_output=False),
        CALENDAR_COLUMNS,
    ),
    ("binary", "passthrough", binary_columns),
])

estimators = {
    "Linear Regression": LinearRegression(**MODEL_PARAMS["Linear Regression"]),
    "Random Forest": RandomForestRegressor(**MODEL_PARAMS["Random Forest"]),
    "Gradient Boosting": GradientBoostingRegressor(**MODEL_PARAMS["Gradient Boosting"]),
    # Chuẩn hóa mục tiêu từ train và đổi dự đoán về đơn vị xe/giờ.
    "MLP": TransformedTargetRegressor(
        regressor=MLPRegressor(**MODEL_PARAMS["MLP"]),
        transformer=StandardScaler(),
    ),
}

models = {}
for name, estimator in estimators.items():
    pipeline = Pipeline([
        ("preprocessing", clone(preprocessor)),
        ("model", estimator),
    ])
    models[name] = pipeline.fit(X_train, y_train)

## So sánh MAE, RMSE và R² trên tập validation

In [ ]:
prediction_table = validation[["date_time", TARGET_COLUMN]].copy()
scores = []

for name, model in models.items():
    prediction = model.predict(X_validation)
    if CLIP_NEGATIVE_PREDICTIONS:
        prediction = prediction.clip(min=0)

    prediction_table[name] = prediction
    scores.append({
        "Mô hình": name,
        "MAE": mean_absolute_error(y_validation, prediction),
        "RMSE": mean_squared_error(y_validation, prediction) ** 0.5,
        "R2": r2_score(y_validation, prediction),
    })

results = pd.DataFrame(scores).set_index("Mô hình").sort_values("MAE")
display(results.round({"MAE": 3, "RMSE": 3, "R2": 4}))

## Vẽ và lưu biểu đồ dự đoán

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt

plot_index = pd.date_range(
    start=prediction_table["date_time"].min(),
    periods=HOURS_TO_PLOT,
    freq="h",
)
# Giữ NaN ở các giờ thiếu để biểu đồ ngắt tại khoảng trống.
plot_data = prediction_table.set_index("date_time").reindex(plot_index)

fig, axes = plt.subplots(
    len(models), 1, figsize=(14, 11), sharex=True, sharey=True, layout="constrained"
)
for ax, name in zip(axes, models):
    ax.plot(plot_data.index, plot_data[TARGET_COLUMN], color="#202020", label="Thực tế")
    ax.plot(plot_data.index, plot_data[name], linestyle="--", label="Dự đoán")
    ax.set_title(name, loc="left")
    ax.set_ylabel("Xe/giờ")
    ax.grid(alpha=0.25)
    ax.legend(loc="upper right")

axes[-1].xaxis.set_major_locator(mdates.HourLocator(byhour=[0, 12]))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%d/%m\n%H:%M"))
axes[-1].set_xlabel("Ngày và giờ")
fig.suptitle("Lưu lượng thực tế và dự đoán của bốn mô hình — Cấu hình A")

output_dir = project_dir / "experiments" / "11-11_Ridge_ABC_phuc" / "img"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"config_a_4_models_{HOURS_TO_PLOT}h.png"
fig.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()